Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

Load and Clean Dataset

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df = df.dropna(subset=["TotalCharges"])

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Define Features and Target

In [3]:
X = df.drop(columns=["Churn", "customerID"])
# Note: 'customerID' is an identifier, not a meaningful customer behavior feature
y = df["Churn"]

Convert Target Variable to Numeric Type

In [4]:
# use a map to convert all "No" to 0 and all "Yes" to 1
y = y.map({
    "No": 0,
    "Yes": 1
})

In [5]:
# check the first five rows of y's distribution
y.head() # 0 = did not churn, 1 = churned

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

Split Dataset into Train/Test Components

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Identify Numerical vs. Categorical Features

In [7]:
# these features' data consists of numbers
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [8]:
# these features' data consist of discrete categories
categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

Build Preprocessing Pipeline

In [9]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# The Imputer replaces missing numerical values with the median of that column
# StandardScaler transforms each numerical feature using z-score, i.e. it is approximately centered around 0 with standard deviation 1.
# The formula is: z = (x - μ​) / σ
# This is useful for Logistic Regression because features on very different scales can affect optimization.

In [10]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Note: handle_unknown="ignore" ensures that if the API receives a category not seen during training, the encoder won't crash.

Combine using ColumnTransformer

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Note: ColumnTransfer applies different transformations to different columns.
# Specifically, the outputs are combined into one feature matrix.

Run Preprocessing Test

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
# Here, preprocessor learns information from training data and transforms it.
X_test_processed = preprocessor.transform(X_test)
# Meanwhile, here, the preprocessor transforms the test data using the information learned from training.

In [13]:
# Inspect the result
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (5625, 19)
Processed training shape: (5625, 45)
Processed test shape: (1407, 45)


Save the Preprocessing Pipeline

In [14]:
import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

# This saves the fitted preprocessing object.
# Later, the API will use the same preprocessing steps to transform new customer data.

['../models/preprocessor.joblib']